<a href="https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/daniausman24-bot/ML_Internship_Track"
REPO_DIR = "ML_Internship_Track"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/ML_Internship_Track
Starter data found. You're ready.


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniausman24-bot/ML_Internship_Track/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Type: Ranking / scoring.

This isn't a plain yes/no classification — the decision an editor needs is
"which page first," not "is this page declining or not." A binary label
tells them almost nothing about priority. What they need is every declining
page assigned a priority score, then sorted, so they can work down the list
until they run out of time. That's ranking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df["trend_pct"].describe())
print(df[df["trend_direction"] == "down"]["trend_pct"].describe())

count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64
count    16262.000000
mean       -58.113830
std         23.488605
min       -100.000000
25%        -75.900000
50%        -55.600000
75%        -38.500000
max        -20.000000
Name: trend_pct, dtype: float64


## 2. Target or proxy

Target: is_declining_label, defined as trend_direction == "down".

Where it comes from: trend_direction is itself computed from trend_pct, a
measured 90-day change in traffic/clicks — so this is an OBSERVED outcome
(a real measured change), not a human-defined rule like "flag anything
under 500 words." That matters: because the label is derived from
trend_pct, trend_pct and trend_direction can never be used as model
features, or the model would just learn to read its own label back.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df[["trend_pct", "trend_direction"]].drop_duplicates().sort_values("trend_pct").head(10))
print(df[["trend_pct", "trend_direction"]].drop_duplicates().sort_values("trend_pct").tail(10))

       trend_pct trend_direction
23        -100.0            down
5013       -99.8            down
13100      -99.7            down
6263       -99.6            down
563        -99.5            down
3363       -99.4            down
7557       -99.3            down
2433       -99.2            down
9291       -99.1            down
12096      -99.0            down
       trend_pct trend_direction
27305    11650.0              up
9097     14900.0              up
24726    15825.0              up
19697    17077.8              up
3561     21400.0              up
14549    26166.7              up
15405    27907.3              up
24695    44900.0              up
11           NaN             new
73           NaN            flat


## 3. Success metric

Metric: Precision@50 — of the top 50 pages the model ranks first, how many
are actually declining?

This is defensible because it matches how the output gets used: an editor
works down a list a fixed number at a time, so what matters is how clean
the TOP of that list is, not overall accuracy across all 30,000 pages. The
baseline hand-rule in this repo scores 0.240 on this metric; a random
forest model scores 0.740 — I can defend "good" here as meaningfully
beating that 0.240 baseline, not an arbitrary target picked after the fact.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
report = open("outputs/model_report.md").read()
start = report.find("| Model |")
print(report[start:start+400])

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |

## Final Queue

- High-confidence items: 3,605
- Medium-confidence items


## 4. The unit of analysis, as a real dataframe

One row = one pseudonymized content page belonging to one client. content_id
identifies the page, client_id identifies which of the 32 clients owns it —
both are pseudonyms used only for grouping, never as model features.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_id", "client_id", "trend_direction", "avg_position",
        "clicks_90d", "content_age_days", "days_since_last_update"]
print(df[cols].head())

             content_id          client_id trend_direction  avg_position  \
0  content_304f48230142  client_f369cb89fc            down          10.6   
1  content_a1fb4e703a9e  client_4e07408562            down          20.3   
2  content_9aa793d4d895  client_7f2253d7e2            down          36.5   
3  content_331d6c4de07b  client_19581e27de          stable           6.2   
4  content_d99b7a2d90ca  client_3fdba35f04            down          44.0   

   clicks_90d  content_age_days  days_since_last_update  
0          29               187                      20  
1           7               445                      25  
2          11               141                      20  
3          58               463                      22  
4          24               263                      14  


## 5. Why ML beats a fixed rule here

A fixed if-statement (e.g. "flag pages older than 200 days with falling
CTR") is exactly what the baseline_rules row above tests — and it only
hits 0.240 precision@50. The signals that actually separate declining
pages from stable ones (position, impressions, engagement, freshness, age)
move together in ways a single threshold can't capture — a page can be
old AND high-traffic AND declining, or new AND low-traffic AND stable.
The random forest picks up on days_with_impressions, avg_position, and
content_age_days together, tripling precision@50 to 0.740 — real evidence
the pattern was too tangled for one rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signal_cols = ["avg_position", "content_age_days", "days_since_last_update",
               "ctr", "engagement_rate"]
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df[signal_cols + ["is_declining"]].corr()["is_declining"].sort_values())

content_age_days         -0.163882
ctr                      -0.061911
avg_position             -0.029035
engagement_rate          -0.012743
days_since_last_update    0.081383
is_declining              1.000000
Name: is_declining, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.